# 22 — SmallCNN1D + Data Augmentation Strength Sweep

**目的**: 測定条件のずれを模したデータ拡張でfold間ばらつきが縮むか検証。

**固定**: モデル=SmallCNN1D(~12k params, nb16), 前処理=SNV+SG1(41,3,1),
Adam(1e-3)+CosineAnnealing, batch=32, MSELoss, patience=20, seed=42

**変更**: strength ∈ {0.0, 0.5, 1.0, 2.0} のみ

**パイプライン(学習)**: augment(strength) → SNV → SG1 → StandardScaler → CNN
**パイプライン(検証/テスト)**: SNV → SG1 → StandardScaler → CNN  ← 拡張なし

StandardScaler fit は fold内学習側の非拡張データのみ（リーク防止）。
Fold3(高含水率)の改善は目標にしない。Fold1,2,4,5 のばらつき縮小を主指標とする。

In [ ]:
import sys, os, copy
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from src.utils import load_data, parse_spectra, get_groups, make_submission

SEED       = 42
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLIP_T     = 200.0
STRENGTHS  = [0.0, 0.5, 1.0, 2.0]
STR_LABELS = ['aug0', 'aug05', 'aug10', 'aug20']

torch.manual_seed(SEED)
np.random.seed(SEED)

train_df, test_df = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
test_meta, _, X_test_raw, _ = parse_spectra(test_df)
y      = y_s.values.astype(float)
groups = get_groups(train_meta)
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))

print(f'Device: {DEVICE}')
print(f'Train: {X_raw.shape}  Test: {X_test_raw.shape}')
print(f'Strengths:  {STRENGTHS}')
print(f'Labels:     {STR_LABELS}')
print(f'Total CV runs: {len(STRENGTHS) * len(SPLITS)}  (+{len(STRENGTHS)} test runs)')

In [ ]:
# SmallCNN1D — exact copy from nb16, DO NOT modify
class SmallCNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1,  8,  kernel_size=15, padding=7), nn.BatchNorm1d(8),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(8,  16, kernel_size=9,  padding=4), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=5,  padding=2), nn.BatchNorm1d(32), nn.ReLU(),
            nn.AdaptiveAvgPool1d(8),
        )
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(32 * 8, 32), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        h = self.conv(x.unsqueeze(1))
        return self.fc(h.view(x.size(0), -1)).squeeze(1)

n_params = sum(p.numel() for p in SmallCNN1D().parameters())
print(f'SmallCNN1D params: {n_params:,}')
assert 5000 < n_params < 30000, f'Expected ~12k params, got {n_params}'
print('Model OK')

In [ ]:
def snv_sg1(R):
    """SNV -> SG1(window=41, poly=3, deriv=1). Works on any (N, W) array."""
    A = R.copy().astype(np.float64)
    A = (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)
    A = savgol_filter(A, window_length=41, polyorder=3, deriv=1, axis=1)
    return A.astype(np.float32)


def augment(R, strength=1.0):
    """
    Augment raw spectra before SNV+SG1 (physical ordering).
    Simulates measurement variation:
      (1) Additive Gaussian noise (per-point)
      (2) Baseline shift (global additive offset)
      (3) Linear tilt (wavenumber-direction trend)
      (4) Multiplicative scale (scattering intensity variation)
    All amplitudes are scaled by per-spectrum std.
    """
    out = R.copy().astype(np.float64)
    B, W = out.shape
    s    = out.std(1, keepdims=True) + 1e-8
    ramp = np.linspace(-1, 1, W)[None, :]
    out += np.random.normal(0, 0.002 * strength, (B, W)) * s          # (1)
    out += np.random.normal(0, 0.010 * strength, (B, 1)) * s          # (2)
    out += np.random.normal(0, 0.010 * strength, (B, 1)) * ramp * s   # (3)
    out *= (1 + np.random.normal(0, 0.010 * strength, (B, 1)))        # (4)
    return out.astype(np.float32)


class RawDataset(Dataset):
    """Holds raw spectra for on-the-fly augmentation in the training loop."""
    def __init__(self, X_raw, y):
        self.X = X_raw.astype(np.float32)
        self.y = y.astype(np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


def rmse_all(yt, yp):
    return float(np.sqrt(np.mean((np.asarray(yt) - np.asarray(yp)) ** 2)))

def rmse_le(yt, yp, T=170.0):
    yt, yp = np.asarray(yt), np.asarray(yp)
    m = yt <= T
    return float(np.sqrt(np.mean((yt[m] - yp[m]) ** 2))) if m.sum() > 0 else np.nan

print('Functions defined: snv_sg1 / augment / RawDataset / metrics')

In [ ]:
def train_model_aug(X_raw_tr, ytr, X_raw_va, yva,
                    strength=0.0, n_epochs=100, batch=32, lr=1e-3, patience=20):
    """
    Train SmallCNN1D with on-the-fly augmentation.

    Train   : augment(strength) -> snv_sg1 -> StandardScaler -> CNN
    Val/Test: snv_sg1 -> StandardScaler -> CNN  (no augmentation)

    StandardScaler is fit on non-augmented preprocessed training data only.
    """
    # Fit scaler on clean (non-augmented) training spectra
    sc = StandardScaler()
    sc.fit(snv_sg1(X_raw_tr))

    # Validation preprocessing (no augmentation)
    Xva_s = sc.transform(snv_sg1(X_raw_va)).astype(np.float32)
    Xva_t = torch.from_numpy(Xva_s).to(DEVICE)
    yva_t = torch.from_numpy(yva.astype(np.float32)).to(DEVICE)

    # DataLoader holds raw data for on-the-fly augmentation
    loader = DataLoader(RawDataset(X_raw_tr, ytr), batch_size=batch,
                        shuffle=True, drop_last=False)

    torch.manual_seed(SEED)
    np.random.seed(SEED)
    model = SmallCNN1D().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    crit  = nn.MSELoss()

    best_val   = float('inf')
    best_state = None
    best_preds = None
    no_improve = 0
    loss_log   = []
    stop_ep    = n_epochs

    for epoch in range(n_epochs):
        model.train()
        ep_loss = 0.0
        for xb_raw, yb in loader:
            xb_np = xb_raw.numpy()
            if strength > 0:
                xb_np = augment(xb_np, strength)
            xb_s = sc.transform(snv_sg1(xb_np)).astype(np.float32)
            xb_t = torch.from_numpy(xb_s).to(DEVICE)
            yb_t = yb.to(DEVICE)
            pred = model(xb_t)
            loss = crit(pred, yb_t)
            opt.zero_grad()
            loss.backward()
            opt.step()
            ep_loss += loss.item() * len(yb)
        sched.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(Xva_t)
            val_loss = crit(val_pred, yva_t).item()

        loss_log.append({
            'epoch':      epoch + 1,
            'train_rmse': (ep_loss / len(ytr)) ** 0.5,
            'val_rmse':   val_loss ** 0.5,
        })

        if val_loss < best_val:
            best_val   = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_preds = val_pred.cpu().numpy()
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            stop_ep = epoch + 1
            break

    model.load_state_dict(best_state)
    return model, sc, best_preds, loss_log, stop_ep

print('train_model_aug defined.')
print('Train   : augment(strength) -> snv_sg1 -> StandardScaler -> CNN')
print('Val/Test: snv_sg1 -> StandardScaler -> CNN')

## GroupKFold CV — Augmentation Strength Sweep

**主観察点**: Std(F1,2,4,5) が strength=0 より小さくなるか（頑健性改善の指標）
**注意**: CV mean はLBと相関しないため、数値の大小でモデル選択しない。

In [ ]:
print('=== GroupKFold CV — augmentation sweep ===')
print('(CV = health/robustness check; NOT correlated with LB)')
print()

all_results   = {}   # strength -> [fold_result dicts]
all_oof       = {}   # strength -> (oof_y, oof_p)
all_loss_logs = {}   # strength -> fold1 loss_log (for divergence check)

for strength in STRENGTHS:
    label = STR_LABELS[STRENGTHS.index(strength)]
    print(f'=== strength={strength} ({label}) ===')
    fold_results = []
    oof_y_list, oof_p_list = [], []
    fold1_llog = None

    for fi, (tr, va) in enumerate(SPLITS):
        _, _, best_p, llog, stop_ep = train_model_aug(
            X_raw[tr], y[tr], X_raw[va], y[va], strength=strength)

        r_all = rmse_all(y[va], best_p)
        r_le  = rmse_le(y[va], best_p)
        fold_results.append({
            'fold':       fi + 1,
            'RMSE_all':   round(r_all, 2),
            'RMSE_le170': round(r_le,  2),
            'stop_ep':    stop_ep,
        })
        oof_y_list.append(y[va])
        oof_p_list.append(best_p)
        if fi == 0:
            fold1_llog = llog
        print(f'  Fold {fi+1}: RMSE_le170={r_le:.2f}%  '
              f'RMSE_all={r_all:.2f}%  stop={stop_ep}ep')

    oof_y = np.concatenate(oof_y_list)
    oof_p = np.concatenate(oof_p_list)
    all_results[strength]   = fold_results
    all_oof[strength]       = (oof_y, oof_p)
    all_loss_logs[strength] = fold1_llog

    folds_le = [r['RMSE_le170'] for r in fold_results]
    folds14  = [folds_le[i] for i in [0, 1, 3, 4]]  # Fold3 excluded
    mean_le  = float(np.mean(folds_le))
    std_no3  = float(np.std(folds14))
    print(f'  -> mean={mean_le:.2f}%  std(F1,2,4,5)={std_no3:.2f}%')
    print(f'  -> OOF: min={oof_p.min():.1f}  mean={oof_p.mean():.1f}  '
          f'max={oof_p.max():.1f}  >170:{(oof_p > 170).sum()}')
    print()

print('CV complete.')

In [ ]:
# CV summary table
print('=' * 90)
print('CV Summary — RMSE_le170 (%)  ←健全性チェック用。CV値の大小でモデル選択しない。')
print('=' * 90)
print(f'{"Str":>5} | {"F1":>6} {"F2":>6} {"F3":>6} {"F4":>6} {"F5":>6} '
      f'| {"Mean":>7} {"Std(F1245)":>11} | {"OOF_mean":>9} {">170":>5}')
print('-' * 90)

for strength in STRENGTHS:
    fold_results = all_results[strength]
    oof_y, oof_p = all_oof[strength]
    folds_le = [r['RMSE_le170'] for r in fold_results]
    folds14  = [folds_le[i] for i in [0, 1, 3, 4]]
    mean_le  = float(np.mean(folds_le))
    std_no3  = float(np.std(folds14))
    print(f'{strength:>5.1f} | {folds_le[0]:>6.2f} {folds_le[1]:>6.2f} '
          f'{folds_le[2]:>6.2f} {folds_le[3]:>6.2f} {folds_le[4]:>6.2f} '
          f'| {mean_le:>7.2f} {std_no3:>11.2f} '
          f'| {oof_p.mean():>9.1f} {(oof_p > 170).sum():>5}')

print('=' * 90)
print('Fold3(species 15/17/19): 高含水率で構造的高誤差 -> 頑健性指標から除外。')
print('Std(F1,2,4,5): aug0より小さければ頑健性改善。OOF_mean 40-55%が健全範囲。')

In [ ]:
# Loss curve check — Fold1 per strength (divergence/instability detection)
os.makedirs('../results', exist_ok=True)

fig, axes = plt.subplots(1, len(STRENGTHS), figsize=(16, 4), sharey=True)
for ax, strength in zip(axes, STRENGTHS):
    llog = all_loss_logs[strength]
    eps  = [r['epoch']      for r in llog]
    trs  = [r['train_rmse'] for r in llog]
    vals = [r['val_rmse']   for r in llog]
    ax.plot(eps, trs, label='train', alpha=0.7)
    ax.plot(eps, vals, label='val',   alpha=0.9, lw=1.5)
    ax.set_title(f'strength={strength}\nFold1  stop@{eps[-1]}ep')
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('RMSE (%)')
plt.suptitle('Loss curves — Fold1 per augmentation strength', y=1.02)
plt.tight_layout()
plt.savefig('../results/nb22_loss_curves.png', dpi=110)
plt.close()
print('Saved: results/nb22_loss_curves.png')

## Full Train → Test Predictions (all 4 strengths)

avg_stop は各 strength の CV fold 平均停止 epoch を使用。
test_mean ≈ 40–55% が健全範囲。外れたものは「非推奨」と明記。

In [ ]:
print('=== Full Train -> Test Predictions ===')
os.makedirs('../submissions', exist_ok=True)

te_results = {}

for strength, label in zip(STRENGTHS, STR_LABELS):
    fold_results = all_results[strength]
    avg_stop = int(round(np.mean([r['stop_ep'] for r in fold_results])))
    print(f'\nstrength={strength}  label={label}  avg_stop={avg_stop}ep')

    # Scaler fit on full clean training data
    sc_full = StandardScaler()
    sc_full.fit(snv_sg1(X_raw))

    # Test preprocessing (no augmentation)
    Xte_s = sc_full.transform(snv_sg1(X_test_raw)).astype(np.float32)
    Xte_t = torch.from_numpy(Xte_s).to(DEVICE)

    # Full train with augmentation
    loader_full = DataLoader(RawDataset(X_raw, y), batch_size=32, shuffle=True)
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    model_f = SmallCNN1D().to(DEVICE)
    opt_f   = torch.optim.Adam(model_f.parameters(), lr=1e-3)
    sched_f = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, T_max=100)
    crit_f  = nn.MSELoss()

    for ep in range(avg_stop):
        model_f.train()
        for xb_raw, yb in loader_full:
            xb_np = xb_raw.numpy()
            if strength > 0:
                xb_np = augment(xb_np, strength)
            xb_s = sc_full.transform(snv_sg1(xb_np)).astype(np.float32)
            xb_t = torch.from_numpy(xb_s).to(DEVICE)
            yb_t = yb.to(DEVICE)
            loss = crit_f(model_f(xb_t), yb_t)
            opt_f.zero_grad(); loss.backward(); opt_f.step()
        sched_f.step()

    model_f.eval()
    with torch.no_grad():
        te_pred = model_f(Xte_t).cpu().numpy()
    te_pred_clip = np.clip(te_pred, 0, CLIP_T)

    fname = f'../submissions/sub_cnn_{label}.csv'
    make_submission(test_meta, te_pred_clip, fname)

    te_results[strength] = {
        'label':    label,
        'mean':     float(te_pred_clip.mean()),
        'min':      float(te_pred_clip.min()),
        'max':      float(te_pred_clip.max()),
        'std':      float(te_pred_clip.std()),
        'gt170':    int((te_pred_clip > 170).sum()),
        'avg_stop': avg_stop,
    }
    print(f'  test: min={te_pred_clip.min():.1f}  mean={te_pred_clip.mean():.1f}  '
          f'max={te_pred_clip.max():.1f}  std={te_pred_clip.std():.1f}  '
          f'>170={(te_pred_clip > 170).sum()}')

In [ ]:
# Final summary + recommendation
print()
print('=' * 98)
print('Final Summary')
print('=' * 98)
print(f'{"Str":>5} | {"F1":>6} {"F2":>6} {"F3":>6} {"F4":>6} {"F5":>6} '
      f'| {"Mean":>7} {"Std(F1245)":>11} | {"test_mean":>9} {">170":>5} | Note')
print('-' * 98)

ref_std = float(np.std([all_results[0.0][i]['RMSE_le170'] for i in [0, 1, 3, 4]]))

for strength in STRENGTHS:
    fold_results = all_results[strength]
    tr           = te_results[strength]
    folds_le     = [r['RMSE_le170'] for r in fold_results]
    folds14      = [folds_le[i] for i in [0, 1, 3, 4]]
    mean_le      = float(np.mean(folds_le))
    std_no3      = float(np.std(folds14))
    test_mean    = tr['mean']
    gt170        = tr['gt170']
    healthy      = (30 <= test_mean <= 60) and (gt170 <= 30)
    delta_std    = std_no3 - ref_std
    if not healthy:
        note = 'NON-RECOMMEND (dist anomaly)'
    elif delta_std < -0.5:
        note = f'ROBUST +  (std {delta_std:+.2f} vs aug0)'
    elif delta_std > 0.5:
        note = f'std worse ({delta_std:+.2f} vs aug0)'
    else:
        note = f'no change ({delta_std:+.2f} vs aug0)'
    print(f'{strength:>5.1f} | {folds_le[0]:>6.2f} {folds_le[1]:>6.2f} '
          f'{folds_le[2]:>6.2f} {folds_le[3]:>6.2f} {folds_le[4]:>6.2f} '
          f'| {mean_le:>7.2f} {std_no3:>11.2f} '
          f'| {test_mean:>9.1f} {gt170:>5} | {note}')

print('=' * 98)
print()
print('== 総括 ==')
print(f'基準 (aug0) Std(F1,2,4,5): {ref_std:.2f}%')
print()
print('Q1. fold間ばらつきは縮んだか:')
for strength in STRENGTHS:
    folds_le = [all_results[strength][i]['RMSE_le170'] for i in [0, 1, 3, 4]]
    std_no3  = float(np.std(folds_le))
    label    = STR_LABELS[STRENGTHS.index(strength)]
    delta    = std_no3 - ref_std
    verdict  = '改善' if delta < -0.5 else ('悪化' if delta > 0.5 else '差なし')
    print(f'  strength={strength}: std={std_no3:.2f}%  ({delta:+.2f})  -> {verdict}')

print()
print('Q2. 予測分布は健全か (test_mean 30-60%, >170 少ない):')
for strength in STRENGTHS:
    tr    = te_results[strength]
    label = tr["label"]
    ok    = (30 <= tr["mean"] <= 60) and (tr["gt170"] <= 30)
    print(f'  sub_cnn_{label}.csv: mean={tr["mean"]:.1f}%  >170={tr["gt170"]}  -> {"OK" if ok else "NG"}')

print()
print('Q3. Publicに投げる候補:')
for strength in STRENGTHS:
    tr        = te_results[strength]
    folds_le  = [all_results[strength][i]['RMSE_le170'] for i in [0, 1, 3, 4]]
    std_no3   = float(np.std(folds_le))
    label     = tr['label']
    healthy   = (30 <= tr['mean'] <= 60) and (tr['gt170'] <= 30)
    robust    = std_no3 < ref_std
    if healthy and robust:
        rec = 'RECOMMEND'
    elif healthy:
        rec = 'SUBMIT (dist OK, robustness not improved)'
    else:
        rec = 'NOT RECOMMENDED'
    print(f'  sub_cnn_{label}.csv  str={strength}  '
          f'test_mean={tr["mean"]:.1f}%  std={std_no3:.2f}%  -> {rec}')

print()
print('Reference: nb16 SmallCNN (no aug) LB=?  nb18 ImprovedCNN 3-seed LB=17.73')
print('最終判断はPublic/Private。CVは健全性とばらつき確認のみ。')